In [1]:
import sys
sys.path.append("/project/src")

In [ ]:
from coxmodel import CoxPH
from lifelines.utils import concordance_index
from lifelines.exceptions import ConvergenceWarning
from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wandb
import joblib
import warnings
from scipy.stats import randint, uniform
from sklearn.utils.class_weight import compute_sample_weight
from scipy.stats import uniform, loguniform
from sklearn.model_selection import validation_curve
from sklearn.pipeline import Pipeline

from preprocessing import (
    split_features_target,
    build_survival_target,
    low_missingness_complete_case_analysis,
    concat_features_target,
    decode_preprocessed_feature_name,
    SURVIVAL_EVENT_COL,
    SURVIVAL_TIME_COL,
    BaseDatasetPreprocessor,
)

# Read datasets

In [ ]:
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive # type: ignore
    drive.mount('/content/drive')
    train_df_csv = "/content/drive/MyDrive/bachelor/nacc_train_reduced.csv"
else:
    train_df_csv = "./data/nacc_train_reduced.csv"

In [4]:
train_df = pd.read_csv(train_df_csv, delimiter=',')

In [5]:
train_df.shape

(6581, 333)

## Compute X and y and calculate weights

In [6]:
train_X, train_y = split_features_target(train_df)

# Initialize wandb

In [7]:
features_num = train_df.shape[1]
n_samples = train_df.shape[0]

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="semariik",
    # Set the wandb project where this run will be logged.
    project="survival-analysis-mci",
)

wandb.define_metric("cox/*", step_metric="cox/lambda")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: mariik04 (semariik) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Parameters and constents definition

In [8]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [9]:
duration_col = "TIME"
event_col    = "EVENT_MCI"
lambdas      = np.logspace(-1, 2, 50)

In [ ]:
param_distributions = {
    'model__penalizer': loguniform(0.1, 100),
    'model__l1_ratio':  uniform(0, 1),
}

In [ ]:
cox_tunning_pipeline = Pipeline([
  ('preprocessor', BaseDatasetPreprocessor()),
  ("model", CoxPH(duration_col=SURVIVAL_TIME_COL, event_col=SURVIVAL_EVENT_COL, compute_weights=True))
])

# Cox model tunning

## Tune parameters with GridSearchCV

In [11]:
def cox_concordance_scorer(estimator, X, y):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        score = estimator.score(X, y)

    for w in caught:
        msg = str(w.message)
        if "Newton-Raphson failed" in msg or "suspiciously close to 0" in msg:
            return 0.5

    return score

In [ ]:
cv_splits = list(cv.split(train_X, train_y[event_col].astype(int)))

random_search = RandomizedSearchCV(
    estimator=cox_tunning_pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    cv=cv_splits,
    scoring=cox_concordance_scorer,
    refit=False,
)

In [13]:
random_search.fit(train_X, train_y)


Fitting CoxPH with parameters: penalizer=15.806558668612091, l1_ratio=0.6105411632339474, compute_weights=True

Fitting CoxPH with parameters: penalizer=15.806558668612091, l1_ratio=0.6105411632339474, compute_weights=True


/usr/local/lib/python3.13/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['categorical__CVPACDEF_CVPACDEF_2.0'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
/usr/local/lib/python3.13/site-packages/lifelines/utils/__init__.py:797: RuntimeWarning: invalid value encountered in divide
  return (X - mean) / std
/usr/local/lib/python3.13/site-packages/lifelines/fitters/coxph_fitter.py:1530: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.0.
  inv_h_dot_g_T = spsolve(-h, g, assume_a="pos", check_finite=False)
/usr/local/lib/python3.13/site-packages/lifelines/utils/__init__.py:1120: ConvergenceWarning: Column categorical__CDRGLOB_CDRGLOB_1.0 have very low variance when conditioned on death event present or not. This may harm convergence.


Fitting CoxPH with parameters: penalizer=15.806558668612091, l1_ratio=0.6105411632339474, compute_weights=True

Fitting CoxPH with parameters: penalizer=59.78999142641633, l1_ratio=0.015045880718517646, compute_weights=True

Fitting CoxPH with parameters: penalizer=59.78999142641633, l1_ratio=0.015045880718517646, compute_weights=True


/usr/local/lib/python3.13/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['categorical__CVPACDEF_CVPACDEF_2.0'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
/usr/local/lib/python3.13/site-packages/lifelines/utils/__init__.py:797: RuntimeWarning: invalid value encountered in divide
  return (X - mean) / std
/usr/local/lib/python3.13/site-packages/lifelines/fitters/coxph_fitter.py:1530: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.0.
  inv_h_dot_g_T = spsolve(-h, g, assume_a="pos", check_finite=False)
/usr/local/lib/python3.13/site-packages/lifelines/utils/__init__.py:1120: ConvergenceWarning: Column categorical__CDRGLOB_CDRGLOB_1.0 have very low variance when conditioned on death event present or not. This may harm convergence.


Fitting CoxPH with parameters: penalizer=59.78999142641633, l1_ratio=0.015045880718517646, compute_weights=True

Fitting CoxPH with parameters: penalizer=1.3003579410345083, l1_ratio=0.4417568297979604, compute_weights=True

Fitting CoxPH with parameters: penalizer=1.3003579410345083, l1_ratio=0.4417568297979604, compute_weights=True


...


Fitting CoxPH with parameters: penalizer=28.036488230375674, l1_ratio=0.20389126685567782, compute_weights=True

Fitting CoxPH with parameters: penalizer=1.2198728050879883, l1_ratio=0.18715396328777212, compute_weights=True

Fitting CoxPH with parameters: penalizer=1.2198728050879883, l1_ratio=0.18715396328777212, compute_weights=True


/usr/local/lib/python3.13/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['categorical__CVPACDEF_CVPACDEF_2.0'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
/usr/local/lib/python3.13/site-packages/lifelines/utils/__init__.py:797: RuntimeWarning: invalid value encountered in divide
  return (X - mean) / std
/usr/local/lib/python3.13/site-packages/lifelines/fitters/coxph_fitter.py:1530: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.0.
  inv_h_dot_g_T = spsolve(-h, g, assume_a="pos", check_finite=False)
/usr/local/lib/python3.13/site-packages/lifelines/utils/__init__.py:1120: ConvergenceWarning: Column categorical__CDRGLOB_CDRGLOB_1.0 have very low variance when conditioned on death event present or not. This may harm convergence.


Fitting CoxPH with parameters: penalizer=1.2198728050879883, l1_ratio=0.18715396328777212, compute_weights=True


/usr/local/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:490: FitFailedWarning: 
50 fits failed out of a total of 150.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
50 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/project/src/coxmodel.py", line 24, in fit
    return super().fit(df, duration_col=self.duration_col, event_col=self.event_col, **kwargs)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/p

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",<lifelines.CoxPH>
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'l1_ratio': <scipy.stats....xffff41515310>, 'penalizer': <scipy.stats....xffff43ced400>}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",<function cox...xffff41394ae0>
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",False
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv``

## Asses overfitting and underfitting and select optimal lambda

Create error curves for cross validation and train. Check that optimally selected parameters are the optimal point,w here no overfit no undefit. Reselect better lambda, in case of need by graph

In [14]:
# train_errors = []
# val_errors   = []
# val_stds     = []

# for penalizer in lambdas:
#     fold_train = []
#     fold_val   = []

#     for train_idx, val_idx in cv_splits:
#         X_train_fold = train_X.iloc[train_idx]
#         y_train_fold = train_y[train_idx]
#         X_val_fold   = train_X.iloc[val_idx]
#         y_val_fold   = train_y[val_idx]

#         model = CoxPH(penalizer=penalizer, l1_ratio=0.0, duration_col=duration_col, event_col=event_col, compute_weights=True)
#         model.fit(X_train_fold, y_train_fold)

#         c_index_train = model.score(X_train_fold, y_train_fold)
#         c_index_val   = model.score(X_val_fold,   y_val_fold)

#         fold_train.append(1 - c_index_train)
#         fold_val.append(1 - c_index_val)

#     train_errors.append(np.mean(fold_train))
#     val_errors.append(np.mean(fold_val))
#     val_stds.append(np.std(fold_val) / np.sqrt(5))

#     wandb.log({
#         "cox/lambda":      penalizer,
#         "cox/log_lambda":  np.log(penalizer),
#         "cox/train_error": train_errors[-1],
#         "cox/val_error":   val_errors[-1],
#         "cox/gap":         val_errors[-1] - train_errors[-1],
#     })

In [15]:
random_search.best_params_

{'l1_ratio': np.float64(0.6105411632339474),
 'penalizer': np.float64(15.806558668612091)}

In [ ]:
best_model_pipeline = Pipeline([
  ('preprocessor', BaseDatasetPreprocessor()),
  ("model", CoxPH(duration_col=SURVIVAL_TIME_COL, event_col=SURVIVAL_EVENT_COL, compute_weights=True, penalizer=random_search.best_params_['model__penalizer'], l1_ratio=random_search.best_params_['model__l1_ratio']))
])

In [ ]:
train_scores, val_scores = validation_curve(
    estimator=best_model_pipeline,
    X=train_X,
    y=train_y,
    param_name='model__penalizer',
    param_range=lambdas,
    cv=cv_splits,
    scoring=cox_concordance_scorer,
)

train_errors = 1 - np.mean(train_scores, axis=1)
val_errors   = 1 - np.mean(val_scores,   axis=1)
val_stds     = np.std(val_scores, axis=1) / np.sqrt(3)


Fitting CoxPH with parameters: penalizer=0.1, l1_ratio=0.6105411632339474, compute_weights=True

Fitting CoxPH with parameters: penalizer=0.11513953993264472, l1_ratio=0.6105411632339474, compute_weights=True

Fitting CoxPH with parameters: penalizer=0.13257113655901093, l1_ratio=0.6105411632339474, compute_weights=True

Fitting CoxPH with parameters: penalizer=0.15264179671752334, l1_ratio=0.6105411632339474, compute_weights=True

Fitting CoxPH with parameters: penalizer=0.1757510624854792, l1_ratio=0.6105411632339474, compute_weights=True

Fitting CoxPH with parameters: penalizer=0.20235896477251572, l1_ratio=0.6105411632339474, compute_weights=True

Fitting CoxPH with parameters: penalizer=0.2329951810515372, l1_ratio=0.6105411632339474, compute_weights=True

Fitting CoxPH with parameters: penalizer=0.2682695795279726, l1_ratio=0.6105411632339474, compute_weights=True

Fitting CoxPH with parameters: penalizer=0.3088843596477481, l1_ratio=0.6105411632339474, compute_weights=True

Fi

/usr/local/lib/python3.13/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['categorical__CVPACDEF_CVPACDEF_2.0'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
/usr/local/lib/python3.13/site-packages/lifelines/utils/__init__.py:797: RuntimeWarning: invalid value encountered in divide
  return (X - mean) / std
/usr/local/lib/python3.13/site-packages/lifelines/fitters/coxph_fitter.py:1530: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.0.
  inv_h_dot_g_T = spsolve(-h, g, assume_a="pos", check_finite=False)
/usr/local/lib/python3.13/site-packages/lifelines/utils/__init__.py:1120: ConvergenceWarning: Column categorical__CDRGLOB_CDRGLOB_1.0 have very low variance when conditioned on death event present or not. This may harm convergence.


Fitting CoxPH with parameters: penalizer=0.11513953993264472, l1_ratio=0.6105411632339474, compute_weights=True


/usr/local/lib/python3.13/site-packages/lifelines/utils/__init__.py:797: RuntimeWarning: invalid value encountered in divide
  return (X - mean) / std
/usr/local/lib/python3.13/site-packages/lifelines/fitters/coxph_fitter.py:1530: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.0.
  inv_h_dot_g_T = spsolve(-h, g, assume_a="pos", check_finite=False)
/usr/local/lib/python3.13/site-packages/lifelines/utils/__init__.py:1120: ConvergenceWarning: Column categorical__CDRGLOB_CDRGLOB_1.0 have very low variance when conditioned on death event present or not. This may harm convergence. This could be a form of 'complete separation'. For example, try the following code:

>>> events = df['EVENT_MCI'].astype(bool)
>>> print(df.loc[events, 'categorical__CDRGLOB_CDRGLOB_1.0'].var())
>>> print(df.loc[~events, 'categorical__CDRGLOB_CDRGLOB_1.0'].var())

A very low variance means that the column categorical__CDRGLOB_CDRGLOB_1.0 completely determines whether a subject dies or not.


Fitting CoxPH with parameters: penalizer=0.13257113655901093, l1_ratio=0.6105411632339474, compute_weights=True


...


Fitting CoxPH with parameters: penalizer=86.85113737513521, l1_ratio=0.6105411632339474, compute_weights=True


/usr/local/lib/python3.13/site-packages/lifelines/utils/__init__.py:797: RuntimeWarning: invalid value encountered in divide
  return (X - mean) / std
/usr/local/lib/python3.13/site-packages/lifelines/fitters/coxph_fitter.py:1530: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.0.
  inv_h_dot_g_T = spsolve(-h, g, assume_a="pos", check_finite=False)
/usr/local/lib/python3.13/site-packages/lifelines/utils/__init__.py:1120: ConvergenceWarning: Column categorical__CDRGLOB_CDRGLOB_1.0 have very low variance when conditioned on death event present or not. This may harm convergence. This could be a form of 'complete separation'. For example, try the following code:

>>> events = df['EVENT_MCI'].astype(bool)
>>> print(df.loc[events, 'categorical__CDRGLOB_CDRGLOB_1.0'].var())
>>> print(df.loc[~events, 'categorical__CDRGLOB_CDRGLOB_1.0'].var())

A very low variance means that the column categorical__CDRGLOB_CDRGLOB_1.0 completely determines whether a subject dies or not.


Fitting CoxPH with parameters: penalizer=100.0, l1_ratio=0.6105411632339474, compute_weights=True


/usr/local/lib/python3.13/site-packages/lifelines/utils/__init__.py:797: RuntimeWarning: invalid value encountered in divide
  return (X - mean) / std
/usr/local/lib/python3.13/site-packages/lifelines/fitters/coxph_fitter.py:1530: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.0.
  inv_h_dot_g_T = spsolve(-h, g, assume_a="pos", check_finite=False)
/usr/local/lib/python3.13/site-packages/lifelines/utils/__init__.py:1120: ConvergenceWarning: Column categorical__CDRGLOB_CDRGLOB_1.0 have very low variance when conditioned on death event present or not. This may harm convergence. This could be a form of 'complete separation'. For example, try the following code:

>>> events = df['EVENT_MCI'].astype(bool)
>>> print(df.loc[events, 'categorical__CDRGLOB_CDRGLOB_1.0'].var())
>>> print(df.loc[~events, 'categorical__CDRGLOB_CDRGLOB_1.0'].var())

A very low variance means that the column categorical__CDRGLOB_CDRGLOB_1.0 completely determines whether a subject dies or not.


Fitting CoxPH with parameters: penalizer=0.1, l1_ratio=0.6105411632339474, compute_weights=True

Fitting CoxPH with parameters: penalizer=0.11513953993264472, l1_ratio=0.6105411632339474, compute_weights=True

Fitting CoxPH with parameters: penalizer=0.13257113655901093, l1_ratio=0.6105411632339474, compute_weights=True

Fitting CoxPH with parameters: penalizer=0.15264179671752334, l1_ratio=0.6105411632339474, compute_weights=True

Fitting CoxPH with parameters: penalizer=0.1757510624854792, l1_ratio=0.6105411632339474, compute_weights=True

Fitting CoxPH with parameters: penalizer=0.20235896477251572, l1_ratio=0.6105411632339474, compute_weights=True

Fitting CoxPH with parameters: penalizer=0.2329951810515372, l1_ratio=0.6105411632339474, compute_weights=True

Fitting CoxPH with parameters: penalizer=0.2682695795279726, l1_ratio=0.6105411632339474, compute_weights=True

Fitting CoxPH with parameters: penalizer=0.3088843596477481, l1_ratio=0.6105411632339474, compute_weights=True

Fi

In [17]:
for i, penalizer in enumerate(lambdas):
    wandb.log({
        "cox/lambda":      penalizer,
        "cox/train_error": train_errors[i],
        "cox/val_error":   val_errors[i],
        "cox/gap":         val_errors[i] - train_errors[i],
    })

In [18]:
train_errors

array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan])

In [19]:
val_errors

array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan])

In [20]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(lambdas, train_errors, color="blue", linewidth=2, label="Train error")
ax.plot(lambdas, val_errors,   color="red",  linewidth=2, label="Validation error (CV)")

ax.fill_between(lambdas, val_errors - val_stds, val_errors + val_stds,
                alpha=0.2, color="red", label="Val ± std")
ax.fill_between(lambdas, train_errors, val_errors,
                alpha=0.15, color="orange", label="Gap (overfitting signal)")

ax.set_xscale("log")
ax.set_xlabel("lambda", fontsize=12)
ax.set_ylabel("Error", fontsize=12)
ax.set_title("Train vs Validation Error — Regularization Diagnostic (Cox)", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("cox_lambda_curves.png", dpi=150, bbox_inches="tight")
plt.show()

wandb.log({"curves/cox_lambda_diagnostic": wandb.Image("cox_lambda_curves.png")})

ValueError: Data has no positive values, and therefore cannot be log-scaled.

Error in callback <function _draw_all_if_interactive at 0xffff45789c60> (for post_execute), with arguments args (),kwargs {}:


ValueError: Data has no positive values, and therefore cannot be log-scaled.

ValueError: Data has no positive values, and therefore cannot be log-scaled.

<Figure size 1000x600 with 1 Axes>

3500 slov

In [ ]:
cox_best_params = random_search.best_params_
cox_best_params

In [ ]:
wandb.log({"best_parameters": cox_best_params})
joblib.dump(cox_best_params, "joblib-storage/cox_best_params.joblib")
wandb.finish()